## *Published models: neural network potentials for complex aqueous systems*


**In a few words -** In this tutorial, we will retrieve published neural network potentials for complex aqueous systems (ions in solution, confined water, surface-water models). Using tools from the [n2p2](https://compphysvienna.github.io/n2p2/) package, we will see how training and test set performances can be reproduced, how structures used for training can be extracted, and how new molecular dynamics simulations can be run. 

**In greater detail -** Reproducibility is crucial in any form of science; neural network models, which typically depend on many parameters and hyperparameters, make it especially difficult for scientists to rapidly and unambiguously reproduce published results. It is therefore mandatory that we adopt standardized procedures to publish machine-learned models, such as neural network potentials, to maximize reproducibility. In this tutorial, we will retrieve models and datasets from a [preprint](https://arxiv.org/abs/2106.00048). The data is made available through a [git repository](https://github.com/water-ice-group/simple-MLP). 

**Tools used -** [n2p2](https://compphysvienna.github.io/n2p2/), a package for neural network potentials. [cp2k](https://www.cp2k.org/), a general molecular dynamics package. 

**How to use this?** This tutorial uses Python, some familiarity with programming would be beneficial but is not required. Run each code cell in order by clicking "Shift + Enter". Feel free to modify the code, or change queries to familiarize yourself with the workings on the code.

Suggested modifications and exercises are included in <font color=blue> blue</font>.

**Outline:**

1. Downloading the data
2. Reproducing published reference vs. model performance tests
3. Extracting and visualizing systems from the training set
4. Running molecular dynamics

**Get started:** Click "Shift-Enter" on the code cells to run! 

**Setup (Google Colab).** Run the next cell first, and again whenever the Colab runtime restarts. It downloads the course files (or uses `TP-MLIP.zip` if you uploaded it in the Files panel on the left), installs n2p2, CP2K and the Python packages we need (1 to 2 minutes the first time), and moves into this tutorial's folder.

In [ ]:
ZIP_URL = 'https://github.com/zacway/TP-MLIP/releases/latest/download/TP-MLIP.zip'

import os
if not os.path.exists('/content/TP-MLIP'):
    # a TP-MLIP.zip uploaded by hand (Files panel on the left) is used instead of downloading it
    if not os.path.exists('/content/TP-MLIP.zip'):
        !curl -Lsf -o /content/TP-MLIP.zip.part "$ZIP_URL" && mv /content/TP-MLIP.zip.part /content/TP-MLIP.zip
    assert os.path.exists('/content/TP-MLIP.zip'), 'Could not download TP-MLIP.zip: upload it with the Files panel on the left, then run this cell again.'
    !unzip -q /content/TP-MLIP.zip -d /content
%run /content/TP-MLIP/colab_setup.py
%cd /content/TP-MLIP/tutorial_03_wsystems

In [ ]:
# First we'll clone the repository:

if not os.path.exists('simple-MLP'):
    !git clone https://github.com/water-ice-group/simple-MLP

We now have a new folder, "simple-MLP", containing, for each system, the model and associated datasets. We'll first try to reproduce atomic force performance plots from the [preprint](https://arxiv.org/abs/2106.00048): see right panels for figures S1 to S6. Reference data (structures: ie. atomic positions, elements, cell vectors, and properties: total energy, and atomic forces) is stored in n2p2-style text files, in the "training-sets" subfolder. These text files follow a standard [format](https://compphysvienna.github.io/n2p2/topics/cfg_file.html). 

In [ ]:
%matplotlib inline

# Here we import tools we'll use later on, and print the the various aqueous systems available. 

import os

import numpy as np

import matplotlib.pyplot as plt

import shutil
import shlex
import subprocess
import glob

from sklearn.metrics import mean_squared_error

au_to_evang = 51.42208619083232
au_to_ev = 27.21138386
au_to_ang = 0.529177

base_dir  = os.getcwd()
mlp_dir   = base_dir+'/simple-MLP'

sys_names = os.listdir(mlp_dir+'/models')

print('Reported aqueous systems are: {}'.format(sys_names))

In [ ]:
# This is a function which can be called to compute model forces
# based on the reported training set, for a given system. 
# It will plot reference vs. model forces, and print the RMSE. 

def make_predictions(sys_name, n_mpi, Keep_files=False, binning=False):
    if not any(sys_name == s for s in sys_names):
        raise Exception('This is not a recognized system name. ')
    
    sys_dir = base_dir+'/predict_'+sys_name
    # First get the number of committee members
    list_mod = os.listdir(mlp_dir+'/models/'+sys_name)
    n_mod = len(list_mod)
    
    print('There are {} committee members in the model. '.format(n_mod))
    
    if os.path.exists(sys_dir) and os.path.isdir(sys_dir):
        shutil.rmtree(sys_dir)
    
    os.chdir(base_dir)
    os.mkdir(sys_dir)
    
    line_n2p2 = 'mpirun -np '+str(n_mpi)+' nnp-dataset 0'
    
    mod_f = 0
    
    co=1
    for mod in list_mod:
        print ('Committee member number {}'.format(co))
        
        co_dir = sys_dir+'/'+mod
        os.mkdir(co_dir)

        # Copy training set
        shutil.copyfile(mlp_dir+'/training-sets/'+sys_name+'/input.data', co_dir+'/input.data')
        
        # Copy model
        shutil.copytree(mlp_dir+'/models/'+sys_name+'/'+mod, co_dir, dirs_exist_ok=True)
        
        # Call n2p2 to compute model energies and forces
        os.chdir(co_dir)
        print('Compute energies and forces... ')
        call_n2p2 = subprocess.run(shlex.split(line_n2p2), universal_newlines=True, capture_output=True, check=True)
        call_n2p2
        print('Done. ')
        
        # Grab the forces
        tmp = np.loadtxt('forces.comp', usecols=(3))
        mod_f += tmp
        
        if co == 1: 
            # Get the reference forces too
            ref_f = np.loadtxt('forces.comp', usecols=(2))
            # As well as the elements array
            ele_ind = np.loadtxt('forces.comp', usecols=())
        
        os.chdir(base_dir)
        co+=1
        
    # Finalize committee forces calculation
    # NB: the committee forces == average of all member forces
        
    mod_f /= n_mod
    
    ref_f *= au_to_evang
    mod_f *= au_to_evang
    
    abs_largest = np.amax(np.absolute(np.concatenate((ref_f, mod_f))))

    # Plot the model vs. reference forces
        
    fig, ax = plt.subplots(1,1)
    
    if binning: 
        hb = ax.hexbin(ref_f, mod_f, gridsize=50, cmap='inferno')
        cb = fig.colorbar(hb, ax=ax)
        cb.set_label('counts')
    else:
        ax.scatter(ref_f, mod_f, color='firebrick', marker='.', alpha=0.5)
    
    x = [abs_largest*-1., abs_largest]
    ax.plot(x,x, linestyle='-', color='gray', alpha=0.2)

    ax.set_xlabel(r'Reference forces (eV/$\AA$)')
    ax.set_ylabel(r'Model forces (eV/$\AA$)')
    
    ax.set_xlim([abs_largest*-1., abs_largest])
    ax.set_ylim([abs_largest*-1., abs_largest])
    
    ax.set_aspect('equal')
    
    print('The RMSE is {:.2f} meV/Ang. '.format(1e3*np.sqrt(mean_squared_error(ref_f, mod_f))))

In [ ]:
# You can select whichever system you like, and see if the predictions
# match what is reported in the preprint. We only compute a global force RMSE
# whereas in the preprint per-element force RMSE's are reported. See next cell. 

make_predictions('cnt-h2o', n_mpi=os.cpu_count(), Keep_files=True, binning=False)

In [ ]:
# Element-wise RMSE are reported in the preprint. We'll need to convert
# that into the global RMSE we compute above, using the following function: 

def element_to_global_rmse(rmse_ele, n_ele):
    # Arguments: rmse_ele, list of floats with per-element RMSEs in meV/Ang
    #            n_ele, list of ints with number of atoms for each element
    # For instance, for the f-h2o system: 
    #            rmse_ele = [55.8, 29.7, 56.8]
    #               n_ele = [1, 128, 64]
    
    # =sqrt((1/193)*(1*55.8**2+128*29.7**2+64*56.8**2))
    
    tmp = 0
    n_tot = 0
    for rmse, ele in zip(rmse_ele, n_ele):
        tmp += ele * rmse**2
        n_tot += ele
        
    print('The corresponding total RMSE is {:.2f} meV/Ang. '.format(np.sqrt(tmp/(n_tot))))

    
# Here's a call to the function using parameters for the cnt-h2o system
# The number of atoms can be extracted from the section "Computational details", subsection "AIMD simulations"
element_to_global_rmse([71.1, 43.1, 97.8], [288, 130, 65])

In [ ]:
# We can also convert training data from an n2p2 format to xyz, for instance. 

# Here we'll use the ase library to manipulate trajectory files. 

import mdtraj as mdt

import ase
import ase.io

import sys
import fileinput

def lines_that_start_with(string, fp):
    return [line for line in fp if line.startswith(string)]

def get_first_frame(sys_name):
    if not any(sys_name == s for s in sys_names):
        raise Exception('This is not a recognized system name. ')
    
    sys_dir = base_dir+'/predict_'+sys_name
    
    # Get the list of elements from input.nn
    with open(sys_dir+'/nnp-001/input.nn', 'r') as fp:
        for line in lines_that_start_with('elements', fp):
            sys_ele = line.split("#", 1)[0].split()[1:]
    
    # Use "nnp-convert" to generate an xyz version of the training data
    line_n2p2 = 'mpirun -np 1 nnp-convert xyz '+' '.join(str(ele) for ele in sys_ele)

    os.chdir(sys_dir+'/nnp-001')
    
    call_n2p2 = subprocess.run(shlex.split(line_n2p2), universal_newlines=True, capture_output=True)
    call_n2p2
    
    os.chdir(base_dir)
    
    # Use ase to grab the first frame, convert the cell/coordinates from au to Ang, 
    # and switch from xyz to pdb. 
    
    tmp = ase.io.read(sys_dir+'/nnp-001/input.xyz')

    tmp.set_cell(au_to_ang*np.asarray(tmp.cell), scale_atoms=True)

    # Save two files, since CP2K can't read 
    # correctly standard CIF files for coordinates apparently
    ase.io.write(sys_dir+'/nnp-001/input.cif', tmp)
    ase.io.write(sys_dir+'/nnp-001/input.pdb', tmp)

    # Load a representation of the system
    
    view = show_md(tmp)
    
    return view

In [ ]:
# ! Make sure you've run the make_predictions() function once on the system you're
# ! interested in before using the following function. 

get_first_frame('cnt-h2o')

**We now have everything we need to use the models for molecular dynamics simulations.**

The following function can be used to generate a custom CP2K input file based on the system you're interested in. Once executed, the cell after it runs the MD in the corresponding subfolder of the "nnp_md" folder. 

In [ ]:
def set_md(sys_name):
    if not any(sys_name == s for s in sys_names):
        raise Exception('This is not a recognized system name. ')
    
    sys_dir = base_dir+'/predict_'+sys_name
    md_dir  = base_dir+'/nnp_md'
    nnp_dir =   md_dir+'/'+sys_name
    
    # Create a subfolder "nnp_md/sys_name". 
    # ! If it already exists, delete it first. 
    if os.path.exists(nnp_dir) and os.path.isdir(nnp_dir):
        shutil.rmtree(nnp_dir)
    
    os.mkdir(nnp_dir)
    
    # Copy template cp2k input
    shutil.copyfile(md_dir+'/template.inp', nnp_dir+'/cp2k.inp')
    
    # Edit the cp2k input file
    for line in fileinput.input(nnp_dir+'/cp2k.inp', inplace=True):
        if line.strip().startswith('MYPATH'):
            line = '@SET PATH '+sys_dir+' \n'
        sys.stdout.write(line)

In [ ]:
# ! Make sure you've run the make_predictions() and gen_first_frame()
# ! functions once on the system you're interested in before using 
# ! the following function. 

set_md('cnt-h2o')

In [ ]:
run_cp2k('cp2k.inp', base_dir+'/nnp_md/cnt-h2o')

You can visualize generated trajectories using the following function: 

_NB: CP2K outputs by default unwrapped XYZ coordinates, and the viewer doesn't wrap atoms back into the periodic box._

In [ ]:
def view_trj(sys_name):
    if not any(sys_name == s for s in sys_names):
        raise Exception('This is not a recognized system name. ')
    
    top = base_dir+'/predict_'+sys_name+'/nnp-001/input.pdb'
    
    md_dir  = base_dir+'/nnp_md'
    nnp_dir =   md_dir+'/'+sys_name
    
    nnp_trj = nnp_dir + '/nnp_md-pos-1.xyz'
    
    view = show_md(mdt.load_xyz(nnp_trj, top=top, stride=1))
    
    return view

In [ ]:
view_trj('cnt-h2o')

**Are these models OK for bulk water?**

All of these models are parameterized for a heterosystem made of water and something else. Can they be used for bulk water too? 

This is a tricky question. First of all, we don't know _a priori_ if the underlying reference model gives a good representation of bulk water (although hybrid DFT + dispersion should be all right). We also don't have computed reference properties for bulk water, so we won't be able to evaluate the quality of the model vs. the reference for that case.

<font color=blue>We can however do a simple preliminary test. Using the previous tutorial's material, try setting up an _NpT_ simulation of bulk water, and monitoring the density as a function of time. Is the density realistic compared to experimental data at room temperature? Are these different models giving the same answer? </font>